# Reducing WRF file sizes 

### Which varibales are constant or zero?

In [34]:
import os
import glob
import xarray as xr
import pandas as pd
import numpy as np
ORIGINAL_DIR='/home/spfm000/evg000/CanESM2_WRF_runs/historical_r1i1p1_1986/WRF/'
# ── Find all original files ───────────────────────────────────────────────────
orig_files = sorted(glob.glob(os.path.join(ORIGINAL_DIR, "wrfout_d01_*")))
print(f"Found {len(orig_files)} original files")

Found 9169 original files


In [35]:
# ============================================================================
#CONSTANCY CHECK — COMPARE FILES FURTHER APART
# ============================================================================

# Pick files far apart in time to avoid "accidentally zero" problem
file1 = xr.open_dataset(orig_files[0])               # first file
file2 = xr.open_dataset(orig_files[len(orig_files)//2])  # middle file
file3 = xr.open_dataset(orig_files[-1])               # last file

print(f"📂 Comparing across simulation:")
print(f"   File 1 (start)  : {os.path.basename(orig_files[0])}")
print(f"   File 2 (middle) : {os.path.basename(orig_files[len(orig_files)//2])}")
print(f"   File 3 (end)    : {os.path.basename(orig_files[-1])}")

results = []

for var_name in sorted(file1.data_vars):
    if var_name not in file2 or var_name not in file3:
        continue

    try:
        vals1 = file1[var_name].values
        vals2 = file2[var_name].values
        vals3 = file3[var_name].values

        # Must be identical across ALL three comparisons
        diff_1_2     = float(np.max(np.abs(vals1 - vals2)))
        diff_1_3     = float(np.max(np.abs(vals1 - vals3)))
        diff_2_3     = float(np.max(np.abs(vals2 - vals3)))
        max_diff     = max(diff_1_2, diff_1_3, diff_2_3)
        is_constant  = np.array_equal(vals1, vals2) and np.array_equal(vals1, vals3)

        # Also flag "accidentally zero" — all values are zero
        all_zero = (np.all(vals1 == 0) and np.all(vals2 == 0) and np.all(vals3 == 0))

        results.append({
            'variable'    : var_name,
            'constant'    : is_constant,
            'all_zero'    : all_zero,
            'max_diff'    : round(max_diff, 6),
            'shape'       : str(file1[var_name].shape),
            'dtype'       : str(file1[var_name].dtype),
        })

    except Exception as e:
        print(f"   ⚠️  {var_name}: {e}")

# ── Results ───────────────────────────────────────────────────────────────────
df = pd.DataFrame(results)

truly_static  = df[(df['constant'] == True) & (df['all_zero'] == False)]
always_zero   = df[(df['constant'] == True) & (df['all_zero'] == True)]
time_varying  = df[df['constant'] == False]

print(f"\n{'='*60}")
print(f"  ✅ Truly static (non-zero)  : {len(truly_static)}")
print(f"  ⚠️  Always zero             : {len(always_zero)}")
print(f"  🔄 Time-varying             : {len(time_varying)}")
print(f"{'='*60}")

print(f"\n✅ TRULY STATIC ({len(truly_static)}) — safe to save once:")
print(truly_static[['variable', 'shape', 'dtype']].to_string(index=False))

print(f"\n⚠️  ALWAYS ZERO ({len(always_zero)}) — check if expected:")
print(always_zero[['variable', 'shape', 'dtype']].to_string(index=False))

print(f"\n🔄 TIME-VARYING ({len(time_varying)}):")
print(time_varying[['variable', 'shape', 'dtype', 'max_diff']].to_string(index=False))

file1.close()
file2.close()
file3.close()

📂 Comparing across simulation:
   File 1 (start)  : wrfout_d01_1985-12-15_12:00:00
   File 2 (middle) : wrfout_d01_1986-06-24_12:00:00
   File 3 (end)    : wrfout_d01_1987-01-01_12:00:00
   ⚠️  Times: ufunc 'subtract' did not contain a loop with signature matching types (dtype('S19'), dtype('S19')) -> None

  ✅ Truly static (non-zero)  : 5
  ⚠️  Always zero             : 17
  🔄 Time-varying             : 113

✅ TRULY STATIC (5) — safe to save once:
variable       shape   dtype
  AREA2D (1, 99, 99) float32
     CF1        (1,) float32
     CF2        (1,) float32
     CF3        (1,) float32
    DX2D (1, 99, 99) float32

⚠️  ALWAYS ZERO (17) — check if expected:
             variable       shape   dtype
               HAILNC (1, 99, 99) float32
  ISEEDARRAY_SPP_CONV      (1, 2)   int32
   ISEEDARRAY_SPP_LSM      (1, 2)   int32
   ISEEDARRAY_SPP_PBL      (1, 2)   int32
ISEEDARR_RAND_PERTURB      (1, 2)   int32
       ISEEDARR_SKEBS      (1, 2)   int32
        ISEEDARR_SPPT      (1, 2)   

### RAINC is zero in d03 because we have a convective permitting run

In [36]:
# Quick sanity check to confirm
print("Checking RAINC vs RAINNC to confirm...")

for var_name in ['RAINC', 'RAINNC', 'RAINSH']:
    means = []
    for f in [orig_files[0], orig_files[len(orig_files)//2], orig_files[-1]]:
        ds   = xr.open_dataset(f)
        vals = ds[var_name].values
        means.append(float(np.nanmean(vals)))
        ds.close()
    
    varying = not all(m == means[0] for m in means)
    print(f"\n  {var_name}:")
    print(f"    start={means[0]:.4f}  mid={means[1]:.4f}  end={means[2]:.4f}")
    print(f"    {'🔄 time-varying' if varying else '✅ constant — expected for CP run'}")

Checking RAINC vs RAINNC to confirm...

  RAINC:
    start=0.0000  mid=37.0210  end=46.1977
    🔄 time-varying

  RAINNC:
    start=0.0000  mid=46.8839  end=48.7053
    🔄 time-varying

  RAINSH:
    start=0.0000  mid=0.0000  end=0.0000
    ✅ constant — expected for CP run


# ============================================================================
## SAVE TRULY STATIC VARIABLES TO A SINGLE REFERENCE FILE
# ============================================================================

In [37]:
OUTPUT_DIR='/gpfs/fs7/dfo/hpcmc/pfm/amh001/DATA/WRF/Compression/'


# Variables worth keeping
static_keep = [
    'AREA2D',
    'DX2D',    # Grid spacing                 — needed for flux calculations
    'CF1',     # \
    'CF2',     #  > Runge-Kutta constants — small, cheap to keep
    'CF3',     # /
]

static_skip = [
    # Always zero — no scientific value
    'HAILNC', 'RAINSH',           # Schemes not active
    'ISEEDARRAY_SPP_CONV',        # \
    'ISEEDARRAY_SPP_LSM',         #  |
    'ISEEDARRAY_SPP_PBL',         #  | Stochastic schemes
    'ISEEDARR_RAND_PERTURB',      #  |  not active
    'ISEEDARR_SKEBS',             #  |
    'ISEEDARR_SPPT',              # /
    'I_ACLWDNT', 'I_ACLWDNTC',   # Unused counters
    'RESM', 'ZETATOP',            # Zero constants
    'SST_INPUT', 'SSTSK',         # SST not active
    'SWNORM',                     # Not computed
    'THIS_IS_AN_IDEAL_RUN',       # \  Config flags
    'SAVE_TOPO_FROM_REAL',        # /  in namelist
]

print(f"Saving {len(static_keep)} static variables to file...")
print(f"Skipping {len(static_skip)} zero/unused variables...")

# ── Load from first file ──────────────────────────────────────────────────────
ds_first = xr.open_dataset(orig_files[0])

# ── Check all requested vars exist ───────────────────────────────────────────
missing = [v for v in static_keep if v not in ds_first]
if missing:
    print(f"⚠️  Variables not found: {missing}")
    static_keep = [v for v in static_keep if v not in missing]

# ── Build static dataset ──────────────────────────────────────────────────────
static_ds = ds_first[static_keep].copy()

# Drop Time dimension where possible (these are truly static)
for var in static_keep:
    if 'Time' in static_ds[var].dims and static_ds[var].shape[0] == 1:
        static_ds[var] = static_ds[var].squeeze('Time', drop=True)
        print(f"   ✅ {var:<10} squeezed Time dimension → {static_ds[var].shape}")

# ── Add metadata ──────────────────────────────────────────────────────────────
static_ds.attrs['description'] = 'Static/constant fields from WRF convection-permitting simulation'
static_ds.attrs['source_file']  = os.path.basename(orig_files[0])
static_ds.attrs['created']      = pd.Timestamp.now().isoformat()
static_ds.attrs['note']         = 'RAINC=0 confirms convective parameterisation was disabled'

# ── Save ──────────────────────────────────────────────────────────────────────
STATIC_FILE = os.path.join(OUTPUT_DIR, 'wrfout_static_fields_d01.nc')

static_ds.to_netcdf(
    STATIC_FILE,
    encoding={var: {'zlib': True, 'complevel': 4} for var in static_keep}
)

print(f"\n✅ Static file saved: {STATIC_FILE}")
print(f"   Size: {os.path.getsize(STATIC_FILE) / 1024**2:.2f} MB")
print(f"\n📋 Contents:")
for var in static_keep:
    print(f"   {var:<10} {str(static_ds[var].shape):<25} {static_ds[var].dtype}")

ds_first.close()

Saving 5 static variables to file...
Skipping 17 zero/unused variables...
   ✅ AREA2D     squeezed Time dimension → (99, 99)
   ✅ DX2D       squeezed Time dimension → (99, 99)
   ✅ CF1        squeezed Time dimension → ()
   ✅ CF2        squeezed Time dimension → ()
   ✅ CF3        squeezed Time dimension → ()

✅ Static file saved: /gpfs/fs7/dfo/hpcmc/pfm/amh001/DATA/WRF/Compression/wrfout_static_fields_d01.nc
   Size: 0.10 MB

📋 Contents:
   AREA2D     (99, 99)                  float32
   DX2D       (99, 99)                  float32
   CF1        ()                        float32
   CF2        ()                        float32
   CF3        ()                        float32
